# Week 2 — Why conversations get expensive, and what to do about it

A bolt-on demo for week 2. Week 2's bot returns a clean typed object every turn. This notebook asks
a different question: **what does that conversation cost on turn 40, and what does it cost when
two hundred thousand people are having it?**

The answer is the reason week 3 exists.

| Section | The point |
|---|---|
| 1 | The model is stateless, so every turn resends the whole conversation |
| 2 | One turn is O(n). A conversation is O(n²). Doubling the turns quadruples the bill |
| 3 | What that is in rupees at your scale — pure arithmetic, no API calls |
| 4 | Three ways out: drop, compact, cache |
| 5 | Dropping is cheapest and it forgets your employee id |
| 6 | Compaction, implemented in 25 lines |

> ~25 API calls on the cheap model. Under two minutes, well under a cent.

---

## Part 0 — Setup

Self-contained: this notebook does not import anything from `week2.ipynb`.

In [1]:
from dotenv import load_dotenv
import os, pathlib, tiktoken
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-4o-mini"
PRICING = {"in": 0.15, "out": 0.60}      # USD per 1M tokens. Check the provider's pricing page.

os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(pathlib.Path.cwd().parent / "tiktoken_cache"))
encoder = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(encoder.encode(text or ""))

def tokens_in(messages):
    """Close enough for teaching. Exact per-message overhead varies by model."""
    return sum(count_tokens(m["content"]) + 4 for m in messages)

def llm(messages):
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

print("client ready")

client ready


---

## Part 1 — Every turn resends the whole conversation

The model has no memory. It does not know that you spoke to it a minute ago. The *only* thing it
knows is the list of messages in this one HTTP request — so to continue a conversation, you resend
all of it, every single turn.

That means the thing you pay for on turn 8 is not the question. It is the seven turns before it.

In [5]:
SYSTEM = ("You are the HR assistant for a software company in Pune. "
          "Answer employee questions about leave, payroll and workplace policy in 2-3 sentences.")

class Chat:
    """The simplest possible conversation: keep every message, resend every message."""

    def __init__(self, system):
        self.messages = [{"role": "system", "content": system}]
        self.history = []   # one {"turn", "input", "output"} entry per ask()

    def ask(self, text):
        self.messages.append({"role": "user", "content": text})
        response = client.chat.completions.create(model=MODEL, messages=self.messages)
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        self.history.append({"turn": len(self.history) + 1,
                             "input": response.usage.prompt_tokens,
                             "output": response.usage.completion_tokens})
        return reply

    def turn_cost(self, entry):
        return (entry["input"] * PRICING["in"] + entry["output"] * PRICING["out"]) / 1_000_000

    def total_cost(self):
        return sum(self.turn_cost(e) for e in self.history)


In [6]:
chat = Chat(SYSTEM)

QUESTIONS = [
    "Hi, I am Priya, employee E-4417, and I work out of the Bangalore office.",
    "How many casual leaves do I get in a year?",
    "And how many sick leaves?",
    "Do unused casual leaves carry over to next year?",
    "How much notice do I have to give before taking leave?",
    "Who approves my leave when my manager is travelling?",
    "Is there a cutoff for applying for leave in December?",
    "Can I take leave during my notice period?",
]


for question in QUESTIONS:
    chat.ask(question)
    e = chat.history[-1]
    print(f"turn {e['turn']}   input: {e['input']:5} tokens   output: {e['output']:4} tokens   "
          f"${chat.turn_cost(e):.6f} this turn   ${chat.total_cost():.6f} so far")

first_turn_tokens = chat.history[0]["input"]
last_turn_tokens = chat.history[-1]["input"]
spent = chat.total_cost()
print(f"\nThe question on turn {len(QUESTIONS)} was a one-liner, same as turn 1 —")
print(f"but it cost {last_turn_tokens / first_turn_tokens:.1f}x as much to ask.")
print(f"Total: {sum(e['input'] for e in chat.history)} input + "
      f"{sum(e['output'] for e in chat.history)} output tokens = ${chat.total_cost():.6f}")


turn 1   input:    62 tokens   output:   32 tokens   $0.000029 this turn   $0.000029 so far
turn 2   input:   113 tokens   output:   42 tokens   $0.000042 this turn   $0.000071 so far
turn 3   input:   169 tokens   output:   47 tokens   $0.000054 this turn   $0.000124 so far
turn 4   input:   234 tokens   output:   40 tokens   $0.000059 this turn   $0.000183 so far
turn 5   input:   294 tokens   output:   46 tokens   $0.000072 this turn   $0.000255 so far
turn 6   input:   358 tokens   output:   48 tokens   $0.000082 this turn   $0.000338 so far
turn 7   input:   425 tokens   output:   54 tokens   $0.000096 this turn   $0.000434 so far
turn 8   input:   496 tokens   output:   44 tokens   $0.000101 this turn   $0.000534 so far

The question on turn 8 was a one-liner, same as turn 1 —
but it cost 8.0x as much to ask.
Total: 2151 input + 353 output tokens = $0.000534


Read the middle column. The questions are all the same size; the *context* is what grows. By the
last turn you are paying to resend an entire conversation in order to add one sentence to it.

Two more things grow with it, and they do not show up in the bill:

- **Latency.** More input tokens means a slower response, on every turn, forever.
- **The ceiling.** Context windows are finite. A long enough conversation stops working, not just
  stops being cheap.

---

## Part 2 — One turn is O(n). The conversation is O(n²)

Turn 1 sends the system prompt. Turn 2 sends the system prompt plus one turn. Turn 40 sends the
system prompt plus thirty-nine turns.

Add those up over a whole conversation and the total is proportional to **n²**, not n:

```
total input tokens  ≈  n × system  +  (n² / 2) × tokens_per_turn
```

So a conversation twice as long does not cost twice as much. It costs roughly **four times** as
much. That single sentence is the whole reason this notebook exists — and the next cell puts money
on it.

---

## Part 3 — What that costs at your scale

No API calls here. This is a spreadsheet, and all four dials are yours to change.

`strategy` is the thing we are about to compare:

- `"full"` — keep every message, resend every message. What Part 1 did.
- `"window"` — keep only the last few turns. Cheapest, and it forgets.
- `"compact"` — replace the old turns with a short summary, keep the recent ones verbatim.

In [7]:
SYSTEM_TOKENS  = count_tokens(SYSTEM)   # resent on every single turn
TURN_TOKENS    = 120                    # one question plus its answer, roughly
SUMMARY_TOKENS = 150                    # what the compacted history collapses to
KEEP_RECENT    = 6                      # turns kept word-for-word

def conversation_input_tokens(turns, strategy):
    """Total input tokens an entire n-turn conversation costs, under one strategy."""
    total = 0
    for older in range(turns):                 # turns already in the history
        if strategy == "full":
            history = older * TURN_TOKENS
        elif strategy == "window":
            history = min(older, KEEP_RECENT) * TURN_TOKENS
        else:                                   # compact
            recent = min(older, KEEP_RECENT)
            history = recent * TURN_TOKENS
            if older > recent:                  # something was summarised away
                history += SUMMARY_TOKENS
        total += SYSTEM_TOKENS + history
    return total

def monthly_cost(conversations, turns, strategy):
    return conversations * conversation_input_tokens(turns, strategy) * PRICING["in"] / 1_000_000

In [8]:
SCENARIOS = [
    ("internal HR bot: 2,000 employees, 4 chats each",   8_000, 12),
    ("customer support bot",                           500_000, 20),
    ("customer support bot, longer chats",             500_000, 40),
]

for label, conversations, turns in SCENARIOS:
    print(f"{label}  ({conversations:,} conversations/month, {turns} turns each)")
    for strategy in ["full", "window", "compact"]:
        cost = monthly_cost(conversations, turns, strategy)
        print(f"   {strategy:7} ${cost:10,.2f} / month"
              f"   ({conversation_input_tokens(turns, strategy):,} input tokens per conversation)")
    saved = monthly_cost(conversations, turns, "full") - monthly_cost(conversations, turns, "compact")
    print(f"   -> compaction saves ${saved:,.0f} / month\n")

internal HR bot: 2,000 employees, 4 chats each  (8,000 conversations/month, 12 turns each)
   full    $      9.92 / month   (8,268 input tokens per conversation)
   window  $      7.76 / month   (6,468 input tokens per conversation)
   compact $      8.66 / month   (7,218 input tokens per conversation)
   -> compaction saves $1 / month

customer support bot  (500,000 conversations/month, 20 turns each)
   full    $  1,753.50 / month   (23,380 input tokens per conversation)
   window  $    934.50 / month   (12,460 input tokens per conversation)
   compact $  1,080.75 / month   (14,410 input tokens per conversation)
   -> compaction saves $673 / month

customer support bot, longer chats  (500,000 conversations/month, 40 turns each)
   full    $  7,107.00 / month   (94,760 input tokens per conversation)
   window  $  2,058.00 / month   (27,440 input tokens per conversation)
   compact $  2,429.25 / month   (32,390 input tokens per conversation)
   -> compaction saves $4,678 / month



Three readings, and the middle one is the one worth arguing about:

**1. Look at the last two scenarios.** Same traffic, same bot, conversations twice as long —
and the bill went up roughly **four times**. That is the n² from Part 2, in rupees. If your product
gets *better* and people chat for longer, your costs do not grow with your success, they outrun it.

**2. At the HR-bot scale, compaction saves about a dollar a month.** So do not build it. A 12-turn
internal tool on a cheap model does not have this problem, and the compaction code is something you
would have to test, monitor and debug forever. **The dial that decides is traffic × length, not
fashion.** Measure first — that is the same rule as the rest of week 2.

**3. Compaction costs more than truncation** — `compact` is dearer than `window` in every row,
because you are paying to carry a summary that truncation simply throws away. You pay that
difference on purpose. Part 5 shows what it buys.

> Not counted above: the summarisation calls themselves, and output tokens. The table is input-only,
> because input is where the growth lives.

---

## Part 4 — Three ways out

| Lever | What it does | What it costs you |
|---|---|---|
| **Drop** | Keep the last K turns, throw the rest away | Cheapest. The bot forgets things it was told |
| **Compact** | Summarise the old turns, keep recent ones verbatim | An extra call now and then; a summary is lossy |
| **Cache** | Resend the same prefix, pay less for it | Needs a stable prefix; week 3 |

Drop is one line of code, so people reach for it first. Here is what it actually does.

---

## Part 5 — Dropping is cheap, and it forgets

Priya said who she was on turn 1, two hundred tokens ago. Now ask the same question three ways.

In [ ]:
PROBE = "Remind me - what is my employee id, and which office am I in?"

full_context = chat.messages + [{"role": "user", "content": PROBE}]
window_context = [chat.messages[0]] + chat.messages[-4:] + [{"role": "user", "content": PROBE}]

print(f"FULL HISTORY   ({tokens_in(full_context)} tokens)")
print("  ", llm(full_context), "\n")

print(f"LAST 4 MESSAGES ONLY   ({tokens_in(window_context)} tokens)")
print("  ", llm(window_context))

The cheap one is wrong in the way that matters: it does not say "I forgot", it says it never knew.
Your user does not experience that as a cost optimisation. They experience it as a bot that stopped
paying attention halfway through.

So: throw away the *words*, keep the *facts*. That is compaction.

---

## Part 6 — Compaction, in two pieces

**Piece one: the summariser.** An ordinary LLM call whose prompt is explicit about what must survive.
Note that it folds in the previous summary — otherwise the second compaction quietly loses everything
the first one captured.

In [ ]:
def summarise(messages, previous_summary=None):
    """Collapse old turns into a short summary that keeps the facts."""
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    if previous_summary:
        transcript = f"Summary of the conversation so far: {previous_summary}\n\n{transcript}"

    return llm([
        {"role": "system", "content":
         "Summarise this HR conversation in under 80 words. Keep every concrete fact about the "
         "employee - name, employee id, office, dates, numbers, and anything that was decided. "
         "Drop the pleasantries and the explanations."},
        {"role": "user", "content": transcript},
    ])

**Piece two: the conversation that calls it.** The only new idea is `context()` — the history you
*store* and the context you *send* stop being the same list.

In [ ]:
class CompactingChat:
    """Keeps every message, but sends a summary instead of the old ones."""

    def __init__(self, system, token_budget=700, keep_recent=4):
        self.system = {"role": "system", "content": system}
        self.history = []          # the recent turns, word-for-word
        self.summary = None        # everything older, collapsed
        self.token_budget = token_budget
        self.keep_recent = keep_recent

    def context(self):
        """What actually goes over the wire."""
        messages = [self.system]
        if self.summary:
            messages.append({"role": "system",
                             "content": "Summary of the earlier conversation: " + self.summary})
        return messages + self.history

    def compact(self):
        old = self.history[:-self.keep_recent]
        if not old:
            return
        self.summary = summarise(old, self.summary)
        self.history = self.history[-self.keep_recent:]
        print(f"      [compacted {len(old)} messages into a {count_tokens(self.summary)}-token summary]")

    def ask(self, text):
        self.history.append({"role": "user", "content": text})
        if tokens_in(self.context()) > self.token_budget:
            self.compact()
        reply = llm(self.context())
        self.history.append({"role": "assistant", "content": reply})
        return reply

In [ ]:
compacting_chat = CompactingChat(SYSTEM)

for turn, question in enumerate(QUESTIONS, 1):
    before = tokens_in(compacting_chat.context()) + count_tokens(question)
    compacting_chat.ask(question)
    print(f"turn {turn}   context sent: {before:5} tokens")

print(f"\nplain Chat  on turn {len(QUESTIONS)}: {last_turn_tokens} tokens in context")
print(f"compacting  on turn {len(QUESTIONS)}: {tokens_in(compacting_chat.context())} tokens in context")

Watch the middle column rather than the total. It climbs, drops when compaction fires, then climbs
again — a sawtooth instead of a straight line. That is the whole trick: the context has a **ceiling**
now, so turn 400 costs about what turn 40 does.

And the facts? Ask the question that truncation got wrong.

In [ ]:
print(compacting_chat.ask(PROBE))
print(f"\n(answered from a {count_tokens(compacting_chat.summary)}-token summary, "
      f"not from the original messages)")
print(f"\nsummary: {compacting_chat.summary}")

---

## What to take away

1. **The model is stateless, so conversation cost is quadratic.** Twice the turns is roughly four
   times the money, plus latency, plus a context window you will eventually hit.
2. **Compaction gives the context a ceiling.** Store everything, send a summary plus the recent
   turns. Turn 400 costs about what turn 40 costs.
3. **A summary is lossy, and you choose what it loses.** That summariser prompt is load-bearing:
   "keep every employee id and date" is the difference between a bot that remembers and one that
   confidently doesn't.
4. **Compaction is not free and not always worth it.** Each compaction is an extra call, the summary
   costs tokens on every turn after it, and at 12 turns on a cheap model the whole thing saves you a
   dollar a month. Run Part 3 with your own numbers before you build it.

### Open questions this leaves you

- **When should compaction fire?** On a token budget, as here? Every N turns? When the user changes
  topic?
- **What if the summary is wrong?** It is model output like any other. Nothing validated it.
- **What about the facts that matter later but looked unimportant then?** Summarising is a guess
  about the future.

That last one does not have a summarisation answer. It has a retrieval answer — store the whole
history, and fetch only what this turn needs.

That is week 3.